# exp111 learned PF observation likelihood probe

Train-side smoke audit for calibrated candidate likelihood. This notebook does not create a submission candidate.

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import pandas as pd

from settings import ExperimentPaths, get_nested, load_config
from learned_pf_observation_likelihood_probe import (
    DEFAULT_TRAIN_FEATURE_CACHE,
    DEFAULT_TRAIN_FEATURE_SCHEMA,
    build_required_columns,
    candidate_specs_from_config,
    find_artifact,
    run_learned_pf_observation_likelihood_probe,
)

paths = ExperimentPaths()
config = load_config()
output_dir = Path('/kaggle/working/artifacts') if Path('/kaggle/working').exists() else paths.artifacts_dir
output_dir.mkdir(parents=True, exist_ok=True)

print('experiment:', get_nested(config, 'experiment.name'))
print('route:', get_nested(config, 'experiment.route'))
print('parent:', get_nested(config, 'lineage.parent'))
print('mode:', get_nested(config, 'likelihood.mode'))
print('run_folds:', get_nested(config, 'likelihood.run_folds'))
print('output_dir:', output_dir)

## 2. Input cache check

In [ ]:
candidates = candidate_specs_from_config(config)
required_columns = build_required_columns(config, candidates)
cache_path = get_nested(config, 'data.exp099_train_feature_cache_local')
schema_path = get_nested(config, 'data.exp099_train_feature_schema_local')
cache = find_artifact(DEFAULT_TRAIN_FEATURE_CACHE, cache_path)
schema = find_artifact(DEFAULT_TRAIN_FEATURE_SCHEMA, schema_path)

print('cache:', cache)
print('schema:', schema)
print('candidate columns:', [spec.column for spec in candidates])
print('required column count:', len(required_columns))
preview = pd.read_csv(cache, usecols=required_columns, nrows=5)
display(preview)

## 3. Run likelihood smoke audit

In [ ]:
summary = run_learned_pf_observation_likelihood_probe(
    output_dir=output_dir,
    cache_path=cache_path,
    schema_path=schema_path,
    max_rows=get_nested(config, 'likelihood.max_rows'),
)
print(json.dumps(summary['decision'], indent=2, sort_keys=True))

## 4. Metrics and artifacts

In [ ]:
metrics_path = output_dir / summary['artifacts']['metrics']
topk_path = output_dir / summary['artifacts']['topk_metrics']
calibration_path = output_dir / summary['artifacts']['calibration']
importance_path = output_dir / summary['artifacts']['feature_importance_mean']

metrics = pd.read_csv(metrics_path)
topk = pd.read_csv(topk_path)
calibration = pd.read_csv(calibration_path)
importance = pd.read_csv(importance_path)

display(metrics)
display(topk.head(20))
display(calibration.head(20))
display(importance.head(30))
print('summary:', output_dir / f"{get_nested(config, 'audit.output_prefix')}_summary.json")